# 05 · Full Analysis Walkthrough

## Project Overview
This notebook walks through the whole analysis in one place: trends over time, products, regions and customer segments. Each section ends with the findings in numbers.

**Question:** Where does this retailer make its money, when, and where is growth coming from?

**Answer (headline results):**
- **Four months make the year:** September to December bring in 52% of annual sales.
- **The biggest segment is the slowest-growing:** Consumer is half of revenue but grew +25% from 2015 to 2018, against +85% for Corporate and +77% for Home Office.
- **The business is growing:** sales rose 50% in four years, from $480K to $722K.

## Business Questions
1. What are the sales trends over time?
2. Which products and categories perform best?
3. How does regional performance vary?
4. What customer segments generate the most revenue?
5. Which products are most profitable? (Not answerable with this export, which has no Profit column.)

## Step 1: Import Libraries

We'll use pandas for data manipulation and matplotlib/seaborn for visualizations.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## Step 2: Load the Data

Load the Superstore sales dataset from CSV file.

In [2]:
# Load the dataset
df = pd.read_csv('../data/raw/superstore.csv')

# Display first few rows
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


## Step 3: Explore the Data

Let's understand the structure and content of our dataset.

In [3]:
# Check dataset shape
print(f"Dataset shape: {df.shape[0]} rows and {df.shape[1]} columns\n")

# Display column names and data types
print("Column information:")
df.info()

Dataset shape: 9800 rows and 18 columns

Column information:
<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sa

In [4]:
# Check for missing values
print("Missing values per column:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found!")

Missing values per column:
Postal Code    11
dtype: int64


In [5]:
# Display summary statistics
print("Summary statistics for numerical columns:")
df.describe()

Summary statistics for numerical columns:


,Row ID,Postal Code,Sales
count,9800.000000,9789.000000,9800.000000
mean,4900.500000,55273.322403,230.769059
std,2829.160653,32041.223413,626.651875
min,1.000000,1040.000000,0.444000
25%,2450.750000,23223.000000,17.248000
50%,4900.500000,58103.000000,54.490000
75%,7350.250000,90008.000000,210.605000
max,9800.000000,99301.000000,22638.480000


## Step 4: Clean the Data

Convert date columns to proper datetime format and create additional useful columns.

### Data Cleaning Rationale

**Date Format Handling:**
- The dataset contains `Order Date` and `Ship Date` columns that need to be converted to proper datetime format for time-based analysis
- We implement robust parsing with mixed format support to handle potential inconsistencies in production data
- Error handling ensures the notebook doesn't crash if unexpected date formats are encountered
- Dates in this file are DD/MM/YYYY, so parsing uses `dayfirst=True` — month-first parsing silently scrambles months and shipping delays
- The `errors='coerce'` parameter converts unparseable dates to NaT (Not a Time) rather than raising exceptions

**Production-Ready Safeguards:**
1. **Mixed Format Support**: Handles scenarios where dates may be in different formats (e.g., MM/DD/YYYY, YYYY-MM-DD)
2. **Error Handling**: Gracefully manages parsing failures by coercing invalid dates to NaT
3. **Validation**: Checks for any failed conversions and reports them for data quality monitoring
4. **Derived Features**: Creates additional time-based columns (Year, Month, Quarter) for aggregation and analysis

In [6]:
# Convert date columns to datetime with robust error handling
# Source dates are DD/MM/YYYY — dayfirst=True is required
try:
    # Convert Order Date with mixed format support
    df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')  # source dates are DD/MM/YYYY
    
    # Convert Ship Date with mixed format support  
    df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True, errors='coerce')
    
    # Check for any dates that failed to parse (became NaT)
    order_date_nulls = df['Order Date'].isna().sum()
    ship_date_nulls = df['Ship Date'].isna().sum()
    
    if order_date_nulls > 0:
        print(f"⚠️  Warning: {order_date_nulls} Order Date values could not be parsed and were set to NaT")
    if ship_date_nulls > 0:
        print(f"⚠️  Warning: {ship_date_nulls} Ship Date values could not be parsed and were set to NaT")
    
    # Extract additional date components for analysis
    df['Year'] = df['Order Date'].dt.year
    df['Month'] = df['Order Date'].dt.month
    df['Month Name'] = df['Order Date'].dt.strftime('%B')
    df['Quarter'] = df['Order Date'].dt.quarter
    
    # Calculate shipping delay in days with edge case handling
    # This handles: missing dates (NaT), mixed formats, and negative values
    df['shipping_delay_days'] = (df['Ship Date'] - df['Order Date']).dt.days
    
    # Check for potential data quality issues
    negative_delays = (df['shipping_delay_days'] < 0).sum()
    if negative_delays > 0:
        print(f"⚠️  Warning: {negative_delays} records have negative shipping_delay_days (Ship Date before Order Date)")
    
    print("✓ Data cleaning completed successfully!")
    print(f"\nDate range: {df['Order Date'].min()} to {df['Order Date'].max()}")
    print(f"Total records: {len(df)}")
    print(f"Records with valid dates: {len(df) - order_date_nulls}")
    
except Exception as e:
    print(f"❌ Error during date conversion: {str(e)}")
    print("Please check the date format in your dataset")
    raise

✓ Data cleaning completed successfully!

Date range: 2015-01-03 00:00:00 to 2018-12-30 00:00:00
Total records: 9800
Records with valid dates: 9800


### Understanding `shipping_delay_days`

The `shipping_delay_days` column represents the time elapsed between when a customer placed an order (`Order Date`) and when the order was shipped (`Ship Date`). This metric is calculated as:

```
shipping_delay_days = Ship Date - Order Date
```

**Edge Cases and Data Quality Considerations:**

1. **Negative Values**: If `shipping_delay_days` is negative, this indicates that the `Ship Date` is earlier than the `Order Date`. This is logically impossible in normal business operations and typically suggests:
   - **Data entry errors**: Dates may have been transposed or entered incorrectly
   - **System timezone issues**: Different systems recording dates in different timezones
   - **Backdated entries**: Orders entered into the system after they were shipped
   - **Unusual shipping practices**: Pre-positioned inventory shipped before official order confirmation

2. **Missing Values (NaT/NaN)**: If either the `Order Date` or `Ship Date` is missing or could not be parsed, the `shipping_delay_days` will be `NaN`. These records should be:
   - Flagged for data quality review
   - Excluded from shipping performance analysis
   - Investigated with the data source team

3. **Zero Values**: A `shipping_delay_days` of 0 means the order was shipped on the same day it was ordered, which is possible for:
   - Same-day shipping services
   - Orders placed early in the day
   - Digital products or services

**Business Interpretation:**

- **Typical Range**: For most businesses, shipping delay ranges from 1-7 days depending on the shipping method
- **Performance Metric**: Lower values indicate faster order fulfillment
- **Customer Satisfaction**: Shipping delay directly impacts customer experience and satisfaction
- **Operational Efficiency**: Can identify bottlenecks in warehouse operations or inventory management

When analyzing this metric, always:
- Filter out or investigate records with negative or missing values
- Segment analysis by shipping mode (Standard Class, Second Class, etc.)
- Compare against service level agreements (SLAs)
- Look for trends over time or by product category

---

# Business Question 1: What are the sales trends over time?

Let's analyze how sales have changed over the years and identify seasonal patterns.

In [7]:
# Calculate total sales by year
yearly_sales = df.groupby('Year')['Sales'].sum().reset_index()
yearly_sales['Sales'] = yearly_sales['Sales'].round(2)

print("Total Sales by Year:")
print(yearly_sales)

# Calculate year-over-year growth
yearly_sales['Growth %'] = yearly_sales['Sales'].pct_change() * 100
print("\nYear-over-Year Growth:")
print(yearly_sales[['Year', 'Growth %']].dropna())

Total Sales by Year:
   Year      Sales
0  2015  479856.21
1  2016  459436.01
2  2017  600192.55
3  2018  722052.02

Year-over-Year Growth:
   Year   Growth %
1  2016  -4.255483
2  2017  30.636811
3  2018  20.303396


In [8]:
# Visualize yearly sales trend
plt.figure(figsize=(10, 6))
plt.plot(yearly_sales['Year'], yearly_sales['Sales'], marker='o', linewidth=2, markersize=8)
plt.title('Total Sales by Year', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)
plt.grid(True, alpha=0.3)

# Add value labels on points
for x, y in zip(yearly_sales['Year'], yearly_sales['Sales']):
    plt.text(x, y, f'${y:,.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

In [9]:
# Calculate monthly sales trend (across all years)
monthly_sales = df.groupby('Month')['Sales'].sum().reset_index()

# Create month names for better readability
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_sales['Month Name'] = [month_names[m-1] for m in monthly_sales['Month']]

print("\nTotal Sales by Month (all years combined):")
print(monthly_sales[['Month Name', 'Sales']])


Total Sales by Month (all years combined):
   Month Name        Sales
0         Jan   94291.6296
1         Feb   59371.1154
2         Mar  197573.5872
3         Apr  136283.0006
4         May  154086.7237
5         Jun  145837.5233
6         Jul  145535.6890
7         Aug  157315.9270
8         Sep  300103.4117
9         Oct  199496.2947
10        Nov  350161.7110
11        Dec  321480.1695


In [10]:
# Visualize monthly sales pattern
plt.figure(figsize=(12, 6))
plt.bar(monthly_sales['Month Name'], monthly_sales['Sales'], color='steelblue', alpha=0.8)
plt.title('Sales by Month (All Years Combined)', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

**Key Findings:**
- Sales grew from $480K (2015) to $722K (2018), about 15% a year, after a 4% dip in 2016.
- September to December bring in 52% of annual sales; November alone is 15.5%. January and February bring in just 7%.
- Inventory, staffing and campaigns should be planned around the Q4 peak (38.5% of sales).

---

# Business Question 2: Which products and categories perform best?

Analyze top-performing categories and sub-categories to understand product performance.

In [11]:
# Calculate sales by category
category_sales = df.groupby('Category')['Sales'].sum().reset_index()
category_sales = category_sales.sort_values('Sales', ascending=False)
category_sales['Sales'] = category_sales['Sales'].round(2)

print("Total Sales by Category:")
print(category_sales)

# Calculate percentage of total sales
total_sales = category_sales['Sales'].sum()
category_sales['Percentage'] = (category_sales['Sales'] / total_sales * 100).round(2)
print("\nCategory Sales Percentage:")
print(category_sales[['Category', 'Percentage']])

Total Sales by Category:
          Category      Sales
2       Technology  827455.87
0        Furniture  728658.58
1  Office Supplies  705422.33

Category Sales Percentage:
          Category  Percentage
2       Technology       36.59
0        Furniture       32.22
1  Office Supplies       31.19


In [12]:
# Visualize category sales
plt.figure(figsize=(10, 6))
plt.bar(category_sales['Category'], category_sales['Sales'], color=['#ff9999', '#66b3ff', '#99ff99'])
plt.title('Total Sales by Category', fontsize=16, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)
plt.xticks(rotation=45, ha='right')

# Add value labels on bars
for i, (cat, sales) in enumerate(zip(category_sales['Category'], category_sales['Sales'])):
    plt.text(i, sales, f'${sales:,.0f}', ha='center', va='bottom', fontsize=10)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [13]:
# Analyze top 10 sub-categories
subcategory_sales = df.groupby('Sub-Category')['Sales'].sum().reset_index()
subcategory_sales = subcategory_sales.sort_values('Sales', ascending=False)
top_10_subcategories = subcategory_sales.head(10)

print("\nTop 10 Sub-Categories by Sales:")
print(top_10_subcategories)


Top 10 Sub-Categories by Sales:
   Sub-Category        Sales
13       Phones  327782.4480
5        Chairs  322822.7310
14      Storage  219343.3920
16       Tables  202810.6280
3       Binders  200028.7850
11     Machines  189238.6310
0   Accessories  164186.7000
6       Copiers  146248.0940
4     Bookcases  113813.1987
1    Appliances  104618.4030


In [14]:
# Visualize top 10 sub-categories
plt.figure(figsize=(12, 6))
plt.barh(top_10_subcategories['Sub-Category'], top_10_subcategories['Sales'], color='coral')
plt.title('Top 10 Sub-Categories by Sales', fontsize=16, fontweight='bold')
plt.xlabel('Sales ($)', fontsize=12)
plt.ylabel('Sub-Category', fontsize=12)
plt.gca().invert_yaxis()  # Highest at the top

# Add value labels
for i, (cat, sales) in enumerate(zip(top_10_subcategories['Sub-Category'], top_10_subcategories['Sales'])):
    plt.text(sales, i, f'  ${sales:,.0f}', va='center', fontsize=9)

plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

**Key Findings:**
- Technology leads ($827K, 36.6%), but Furniture ($729K) and Office Supplies ($705K) are close behind.
- Phones (14.5%) and Chairs (14.3%) are the two biggest sub-categories; together they are 29% of all sales.
- The four smallest sub-categories (Art, Envelopes, Labels, Fasteners) add up to under 3% of sales.

---

# Business Question 3: How does regional performance vary?

Compare sales performance across different regions.

In [15]:
# Calculate sales by region
regional_sales = df.groupby('Region')['Sales'].sum().reset_index()
regional_sales = regional_sales.sort_values('Sales', ascending=False)
regional_sales['Sales'] = regional_sales['Sales'].round(2)

print("Total Sales by Region:")
print(regional_sales)

# Calculate percentage contribution
regional_sales['Percentage'] = (regional_sales['Sales'] / regional_sales['Sales'].sum() * 100).round(2)
print("\nRegional Sales Percentage:")
print(regional_sales[['Region', 'Percentage']])

Total Sales by Region:
    Region      Sales
3     West  710219.68
1     East  669518.73
0  Central  492646.91
2    South  389151.46

Regional Sales Percentage:
    Region  Percentage
3     West       31.40
1     East       29.60
0  Central       21.78
2    South       17.21


In [16]:
# Visualize regional sales with a bar chart
plt.figure(figsize=(10, 6))
bars = plt.bar(regional_sales['Region'], regional_sales['Sales'], 
               color=['#8dd3c7', '#fb8072', '#bebada', '#fdb462'])
plt.title('Total Sales by Region', fontsize=16, fontweight='bold')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Sales ($)', fontsize=12)

# Add value labels
for i, (region, sales) in enumerate(zip(regional_sales['Region'], regional_sales['Sales'])):
    plt.text(i, sales, f'${sales:,.0f}', ha='center', va='bottom', fontsize=10)

plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [17]:
# Visualize regional sales with a pie chart
plt.figure(figsize=(10, 8))
colors = ['#8dd3c7', '#fb8072', '#bebada', '#fdb462']
plt.pie(regional_sales['Sales'], labels=regional_sales['Region'], autopct='%1.1f%%',
        startangle=90, colors=colors, textprops={'fontsize': 12})
plt.title('Sales Distribution by Region', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

In [18]:
# Analyze top 10 states by sales
state_sales = df.groupby('State')['Sales'].sum().reset_index()
state_sales = state_sales.sort_values('Sales', ascending=False)
top_10_states = state_sales.head(10)

print("\nTop 10 States by Sales:")
print(top_10_states)


Top 10 States by Sales:
           State        Sales
3     California  446306.4635
30      New York  306361.1470
41         Texas  168572.5322
45    Washington  135206.8500
36  Pennsylvania  116276.6500
8        Florida   88436.5320
11      Illinois   79236.5170
20      Michigan   76136.0740
33          Ohio   75130.3500
44      Virginia   70636.7200


In [19]:
# Visualize top 10 states
plt.figure(figsize=(12, 6))
plt.barh(top_10_states['State'], top_10_states['Sales'], color='teal')
plt.title('Top 10 States by Sales', fontsize=16, fontweight='bold')
plt.xlabel('Sales ($)', fontsize=12)
plt.ylabel('State', fontsize=12)
plt.gca().invert_yaxis()

# Add value labels
for i, (state, sales) in enumerate(zip(top_10_states['State'], top_10_states['Sales'])):
    plt.text(sales, i, f'  ${sales:,.0f}', va='center', fontsize=9)

plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

**Key Findings:**
- West (31.4%) and East (29.6%) bring in 61% of sales; Central is 21.8% and South 17.2%.
- California and New York alone account for 33% of sales.
- The South is both the smallest region and the slowest-growing (+18% from 2015 to 2018, against +70% for the West).

---

# Business Question 4: What customer segments generate the most revenue?

Analyze sales by customer segment to understand our customer base.

In [20]:
# Calculate sales by customer segment
segment_sales = df.groupby('Segment')['Sales'].sum().reset_index()
segment_sales = segment_sales.sort_values('Sales', ascending=False)
segment_sales['Sales'] = segment_sales['Sales'].round(2)

print("Total Sales by Customer Segment:")
print(segment_sales)

# Calculate percentage
segment_sales['Percentage'] = (segment_sales['Sales'] / segment_sales['Sales'].sum() * 100).round(2)
print("\nSegment Sales Percentage:")
print(segment_sales[['Segment', 'Percentage']])

Total Sales by Customer Segment:
       Segment       Sales
0     Consumer  1148060.53
1    Corporate   688494.07
2  Home Office   424982.18

Segment Sales Percentage:
       Segment  Percentage
0     Consumer       50.76
1    Corporate       30.44
2  Home Office       18.79


In [21]:
# Visualize segment sales
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Bar chart
ax1.bar(segment_sales['Segment'], segment_sales['Sales'], 
        color=['#ff6b6b', '#4ecdc4', '#45b7d1'])
ax1.set_title('Total Sales by Customer Segment', fontsize=14, fontweight='bold')
ax1.set_xlabel('Customer Segment', fontsize=11)
ax1.set_ylabel('Sales ($)', fontsize=11)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (seg, sales) in enumerate(zip(segment_sales['Segment'], segment_sales['Sales'])):
    ax1.text(i, sales, f'${sales:,.0f}', ha='center', va='bottom', fontsize=10)

# Pie chart
ax2.pie(segment_sales['Sales'], labels=segment_sales['Segment'], autopct='%1.1f%%',
        startangle=90, colors=['#ff6b6b', '#4ecdc4', '#45b7d1'], textprops={'fontsize': 11})
ax2.set_title('Sales Distribution by Segment', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [22]:
# Analyze average order value by segment
segment_avg = df.groupby('Segment')['Sales'].mean().reset_index()
segment_avg = segment_avg.sort_values('Sales', ascending=False)
segment_avg.columns = ['Segment', 'Average Order Value']
segment_avg['Average Order Value'] = segment_avg['Average Order Value'].round(2)

print("\nAverage Order Value by Segment:")
print(segment_avg)


Average Order Value by Segment:
       Segment  Average Order Value
2  Home Office               243.40
1    Corporate               233.15
0     Consumer               225.07


In [23]:
# Count orders by segment
segment_orders = df.groupby('Segment')['Order ID'].nunique().reset_index()
segment_orders.columns = ['Segment', 'Number of Orders']
segment_orders = segment_orders.sort_values('Number of Orders', ascending=False)

print("\nNumber of Orders by Segment:")
print(segment_orders)


Number of Orders by Segment:
       Segment  Number of Orders
0     Consumer              2537
1    Corporate              1491
2  Home Office               894


**Key Findings:**
- Consumer brings in half of sales ($1.15M, 50.8%), then Corporate ($688K) and Home Office ($425K).
- Average order-line value is similar across segments ($225 to $243), so the gap comes from order volume, not order size.
- Consumer grew only +25% from 2015 to 2018, while Corporate grew +85% and Home Office +77%.

---

# Business Question 5: Which products are most profitable?

Analyze profitability across categories and identify high-margin products.

**Profitability — not in this export.** The canonical dataset (`data/raw/superstore.csv`) has no `Profit` column, so profitability is a next step (see README), not something computed here from a different sample.

---

## Summary of Key Insights

### Sales Trends
- Sales grew 50% in four years ($480K in 2015 to $722K in 2018).
- September to December bring in 52% of annual sales.

### Product Performance
- Technology leads, but all three categories sit between 31% and 37% of sales.
- Phones and Chairs are 29% of sales on their own.

### Regional Analysis
- West and East are 61% of sales; California and New York are 33%.
- The South is the smallest and slowest-growing region.

### Customer Segments
- Consumer is half of revenue but the slowest-growing segment.
- Corporate and Home Office grew three times faster from 2015 to 2018.

---

## Recommendations

1. **Plan the year around September to December:** lock inventory and campaign calendars by August.
2. **Build a business-customer offer:** Corporate and Home Office are the fastest-growing revenue.
3. **Protect the core:** keep Phones and Chairs in stock in California and New York through Q4.
4. **Review the South** before spending more there.

---

## Next Steps

- Add the full Superstore export with Profit to check these sales wins are not margin losses.
- Measure repeat-purchase rates by segment (98% of customers already order more than once).

## Takeaway

The store is growing and seasonal, and its growth is shifting toward business customers. The highest-value next moves are to plan around September to December and to build an offer for Corporate and Home Office customers. Margin is the open question: it needs the full export with Profit.